In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [2]:
from ultralytics import YOLO
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)

model = YOLO("yolov10n.yaml")

print("\n=== YOLOv10n MODEL CHECK ===")
print("Model:", model.model.__class__.__name__)

# Find detection head
detect_layers = []

for i, m in enumerate(model.model.model):
    if "Detect" in m.__class__.__name__:
        detect_layers.append((i, m))

print("\nDetection layers:")
for i, m in detect_layers:
    print(f"Layer {i}: {m.__class__.__name__}")
    print(f"  nc = {getattr(m, 'nc', 'N/A')}")
    print(f"  reg_max = {getattr(m, 'reg_max', 'N/A')}")

# Parameter count
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(
    p.numel() for p in model.model.parameters()
    if p.requires_grad
)

print("\nParameters:")
print(f"Total     : {total_params:,}")
print(f"Trainable : {trainable_params:,}")

# Dummy forward
x = torch.zeros(1, 3, 640, 640)

with torch.no_grad():
    y = model.model(x)

print("\nDummy forward: PASS")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics: 8.4.211

=== YOLOv10n MODEL CHECK ===
Model: DetectionModel

Detection layers:
Layer 23: v10Detect
  nc = 80
  reg_max = 16

Parameters:
Total     : 2,775,520
Trainable : 2,775,504

Dummy forward: PASS


In [3]:
from pathlib import Path
import ultralytics

ULTRA_ROOT = Path(ultralytics.__file__).parent

print("Ultralytics package:", ULTRA_ROOT)
print("\nSearching for YOLOv10 YAML files...")

matches = list(ULTRA_ROOT.rglob("*yolov10*.yaml"))

if matches:
    for p in matches:
        print("FOUND:", p)
else:
    print("❌ No YOLOv10 YAML files found")

print("\nSearching cfg/models directories...")
for p in ULTRA_ROOT.rglob("models"):
    if p.is_dir():
        print("Models directory:", p)

Ultralytics package: /usr/local/lib/python3.13/dist-packages/ultralytics

Searching for YOLOv10 YAML files...
FOUND: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/v10/yolov10s.yaml
FOUND: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/v10/yolov10b.yaml
FOUND: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/v10/yolov10n.yaml
FOUND: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/v10/yolov10x.yaml
FOUND: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/v10/yolov10m.yaml
FOUND: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/v10/yolov10l.yaml

Searching cfg/models directories...
Models directory: /usr/local/lib/python3.13/dist-packages/ultralytics/models
Models directory: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models


In [4]:
from pathlib import Path
import yaml
from ultralytics import YOLO

SRC = Path("/usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/v10/yolov10n.yaml")
DST = Path("/kaggle/working/yolov10n_visdrone_10class.yaml")

# Load official YOLOv10n YAML
with open(SRC, "r") as f:
    cfg = yaml.safe_load(f)

# Modify only the number of classes
cfg["nc"] = 10
cfg["scale"] = "n"

# Save VisDrone 10-class version
with open(DST, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Created:", DST)
print("nc:", cfg["nc"])
print("scale:", cfg["scale"])

# Build model from YAML
model = YOLO(str(DST))

print("\nModel built successfully.")
print("Model parameters:", sum(p.numel() for p in model.model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in model.model.parameters() if p.requires_grad))

# Inspect Detect layer
for i, m in enumerate(model.model.model):
    if "Detect" in m.__class__.__name__:
        print("\nDetect layer:")
        print("Index:", i)
        print("Type:", m.__class__.__name__)
        print("Classes:", getattr(m, "nc", "N/A"))
        print("reg_max:", getattr(m, "reg_max", "N/A"))

# Dummy forward
import torch

x = torch.randn(1, 3, 640, 640)
with torch.no_grad():
    y = model.model(x)

print("\nDummy forward: PASS")

Created: /kaggle/working/yolov10n_visdrone_10class.yaml
nc: 10
scale: n

Model built successfully.
Model parameters: 2710940
Trainable parameters: 2710924

Detect layer:
Index: 23
Type: v10Detect
Classes: 10
reg_max: 16

Dummy forward: PASS


In [5]:
from ultralytics import YOLO
from pathlib import Path

MODEL_YAML = "/kaggle/working/yolov10n_visdrone_10class.yaml"
DATA_YAML = "/kaggle/working/visdrone_final.yaml"

model = YOLO(MODEL_YAML)

print("=" * 60)
print("YOLOv10n VisDrone Baseline Pre-Training Check")
print("=" * 60)

print("Model YAML :", MODEL_YAML)
print("Data YAML  :", DATA_YAML)
print("Pretrained : False")
print("Classes    :", model.model.yaml["nc"])

# Verify Detect head
for i, m in enumerate(model.model.model):
    if m.__class__.__name__ == "v10Detect":
        print("Detect idx :", i)
        print("Detect nc  :", m.nc)
        print("reg_max    :", m.reg_max)

# Parameter count
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(
    p.numel() for p in model.model.parameters() if p.requires_grad
)

print("Parameters :", f"{total_params:,}")
print("Trainable  :", f"{trainable_params:,}")

print("=" * 60)
print("Ready for controlled training.")
print("=" * 60)

YOLOv10n VisDrone Baseline Pre-Training Check
Model YAML : /kaggle/working/yolov10n_visdrone_10class.yaml
Data YAML  : /kaggle/working/visdrone_final.yaml
Pretrained : False
Classes    : 10
Detect idx : 23
Detect nc  : 10
reg_max    : 16
Parameters : 2,710,940
Trainable  : 2,710,924
Ready for controlled training.


In [6]:
from pathlib import Path
import yaml

DATASET_ROOT = Path(
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final"
)

DATA_YAML = Path("/kaggle/working/visdrone_final.yaml")

names = [
    "pedestrian",
    "people",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle",
    "bus",
    "motor",
]

cfg = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 10,
    "names": names,
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Created:", DATA_YAML)
print("\nYAML contents:")
print(DATA_YAML.read_text())

# Verify required directories
print("\nDataset directory checks:")
for split in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    print(
        f"{split.upper():5s} | "
        f"images={img_dir.exists()} | "
        f"labels={label_dir.exists()}"
    )

Created: /kaggle/working/visdrone_final.yaml

YAML contents:
path: /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final
train: train/images
val: val/images
test: test/images
nc: 10
names:
- pedestrian
- people
- bicycle
- car
- van
- truck
- tricycle
- awning-tricycle
- bus
- motor


Dataset directory checks:
TRAIN | images=True | labels=True
VAL   | images=True | labels=True
TEST  | images=True | labels=True


In [7]:
from ultralytics import YOLO
from pathlib import Path
import torch
import ultralytics

MODEL_YAML = "/kaggle/working/yolov10n_visdrone_10class.yaml"
DATA_YAML = "/kaggle/working/visdrone_final.yaml"

PROJECT = "/kaggle/working/YOLO_VisDrone_Benchmark"
RUN_NAME = "YOLOv10n_AdamW_seed42"

print("=" * 70)
print("YOLOv10n | VisDrone | Controlled Baseline")
print("=" * 70)
print("Ultralytics :", ultralytics.__version__)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

print("\nProtocol")
print("  Model       : YOLOv10n")
print("  Classes     : 10")
print("  Dataset     : VisDrone corrected")
print("  Epochs      : 100")
print("  Image size  : 640")
print("  Batch       : 8")
print("  Optimizer   : AdamW")
print("  lr0         : 0.002")
print("  lrf         : 0.01")
print("  Weight decay: 0.0005")
print("  Cosine LR   : True")
print("  Warmup      : 3 epochs")
print("  AMP         : True")
print("  Seed        : 42")
print("  Determinism : True")
print("  Patience    : 20")
print("  Max det     : 1000")
print("  Workers     : 2")
print("  Copy-paste  : 0")
print("  Pretrained  : False")

model = YOLO(MODEL_YAML)

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,

    optimizer="AdamW",
    lr0=0.002,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,

    warmup_epochs=3,
    amp=True,

    seed=42,
    deterministic=True,

    patience=20,
    max_det=1000,

    workers=2,

    copy_paste=0,

    pretrained=False,

    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True
)

print("\n" + "=" * 70)
print("YOLOv10n training completed")
print("=" * 70)
print("Run directory:", f"{PROJECT}/{RUN_NAME}")
print("Best checkpoint:", f"{PROJECT}/{RUN_NAME}/weights/best.pt")

YOLOv10n | VisDrone | Controlled Baseline
Ultralytics : 8.4.211
PyTorch      : 2.11.0+cu130
CUDA         : True
GPU          : Tesla T4

Protocol
  Model       : YOLOv10n
  Classes     : 10
  Dataset     : VisDrone corrected
  Epochs      : 100
  Image size  : 640
  Batch       : 8
  Optimizer   : AdamW
  lr0         : 0.002
  lrf         : 0.01
  Weight decay: 0.0005
  Cosine LR   : True
  Warmup      : 3 epochs
  AMP         : True
  Seed        : 42
  Determinism : True
  Patience    : 20
  Max det     : 1000
  Workers     : 2
  Copy-paste  : 0
  Pretrained  : False
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/work

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      3.49G      2.989      2.846      1.057        558        640: 100% ━━━━━━━━━━━━ 809/809 7.3it/s 1:50
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 5.2it/s 6.8s
                   all        548      38759      0.307      0.106     0.0633     0.0271

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      3.85G      2.815      2.605      1.015        663        640: 100% ━━━━━━━━━━━━ 809/809 7.3it/s 1:50
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 5.4it/s 6.4s
                   all        548      38759      0.336       0.13     0.0769     0.0351

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      3.85G      2.716      2.435     0.9946        598        640: 100% ━━━━━━━━━━━━ 809/809 7.3it/s 1:50
                 Class     Images  Instances      Box(P 